In [2]:
import torch

class SoftThinking:
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        # Use left padding so the end of every prompt is at the same position (-1).
        # This lets us safely use out.logits[:, -1, :] to get the next-token logits
        self.tokenizer.padding_side = "left"
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.device = device or next(model.parameters()).device

        self.vocab_size = model.config.vocab_size
        self.embed_matrix = model.get_input_embeddings().weight  # (V, d)
        # print(model.config.eos_token_id)
        # print(tokenizer.eos_token_id)
        self.eos_ids = model.config.eos_token_id
        self.think_start_id = tokenizer.convert_tokens_to_ids("<think>")
        self.think_end_id = tokenizer.convert_tokens_to_ids("</think>")


    def _top_k(self, probs, k):
        if k is not None and 0 < k < probs.shape[-1]:
            values, indices = torch.topk(probs, k=k, dim=-1)
            filtered = torch.zeros_like(probs)
            filtered.scatter_(-1, indices, values)
            return filtered
        return probs


    def _top_p(self, probs, p):
        if 0 < p <= 1:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumsum = torch.cumsum(sorted_probs, dim=-1)
            # keep the first token that crosses top_p, drop everything after
            sorted_mask = cumsum - sorted_probs >= p
            sorted_probs = sorted_probs.masked_fill(sorted_mask, 0.0)
            filtered = torch.zeros_like(probs).scatter(-1, sorted_idx, sorted_probs)
            return filtered
        return probs



    def _min_p(self, probs, p):
        if 0 < p <= 1:
            threshold = p * probs.max(dim=-1, keepdim=True).values
            filtered = torch.where(probs < threshold, torch.zeros_like(probs), probs)
            return filtered
        return probs



    def _renormalize_distribution(self, probs):
        total = probs.sum(dim=-1, keepdim=True)
        return probs / total.clamp_min(1e-12)


    def _filter_distribution(self, probs, top_k, top_p, min_p):
        filtered_probs = probs
        if top_p is not None:
            filtered_probs = self._top_p(filtered_probs, top_p)
        if top_k is not None:
            filtered_probs = self._top_k(filtered_probs, top_k)
        if min_p is not None:
            filtered_probs = self._min_p(filtered_probs, min_p)
        filtered_probs = self._renormalize_distribution(filtered_probs)
        return filtered_probs


    def _compact_nonzero(self, filtered_probs):
        B, V = filtered_probs.shape

        nonzero = filtered_probs > 0

        # Maximum number of non-zero values across the batch
        K_max = nonzero.sum(dim=-1).max().item()

        # Sort probabilities descending.
        sorted_probs, sorted_indices = torch.sort(filtered_probs, dim=-1, descending=True)
        # Keep only the potentially non-zero part
        sorted_probs = sorted_probs[:, :K_max]
        sorted_indices = sorted_indices[:, :K_max]

        return sorted_probs, sorted_indices


    def _entropy(self, probs):
        safe = torch.where(probs > 0, probs, torch.ones_like(probs))
        return -(probs * torch.log(safe)).sum(dim=-1)

    def _cold_stop_entropy(self, filtered_probs, threshold):
        # track entropy of the (filtered) distribution
        h = self._entropy(filtered_probs)
        stop = h <= threshold
        return stop, h


    def _soft_token_embedding_full(self, filtered_probs):
        # TODO: Not efficient!
        return filtered_probs @ self.embed_matrix

    def _soft_token_embedding_compact(self, pool_probs, pool_indices):
        # [B, K, D]
        candidate_embeddings = self.embed_matrix[pool_indices]
        # [B, K, 1] * [B, K, D]
        concept_token = (pool_probs.unsqueeze(-1) * candidate_embeddings).sum(dim=1)

        return concept_token


    @torch.no_grad()
    def generate(self, prompts, system_prompt=None, max_thinking_tokens=1024, max_answer_tokens=1024,
                 top_k=5, top_p=None, min_p=None, temperature=1.0, cold_stop_threshold=0.1, streak_threshold=64, mode="soft", print_steps=False):
        raw_prompts = list(prompts)
        if isinstance(prompts, str):
            prompts = [prompts]

        def build(p):
            messages = []
            if system_prompt:
                messages.append({"role": "system", "content": system_prompt})
            messages.append({"role": "user", "content": p})
            return self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=True,
            ) + "<think>\n"

        prompts = [build(p) for p in prompts]

        inputs = self.tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)
        B = input_ids.shape[0]

        # Create position IDs for real tokens independently of the amount of
        # left padding, so the first real token starts at position 0.
        position_ids = attention_mask.long().cumsum(-1) - 1
        position_ids.masked_fill_(attention_mask == 0, 1)

        # ---- prefill ----
        out = self.model(input_ids=input_ids, attention_mask=attention_mask, position_ids=position_ids, use_cache=True)

        past_key_values = out.past_key_values
        logits = out.logits[:, -1, :]
        next_position = position_ids[:, -1:] + 1

        num_thinking_tokens = torch.zeros(B, dtype=torch.long, device=self.device)
        num_answer_tokens = torch.zeros(B, dtype=torch.long, device=self.device)

        thinking = torch.ones(B, dtype=torch.bool, device=self.device)
        finished = torch.zeros(B, dtype=torch.bool, device=self.device)
        low_entropy_streak = torch.zeros(B, dtype=torch.long, device=self.device)

        trace = []
        collected_ids = []   # list of (B, 1) tensors, one per step
        collected_mask = []  # list of (B,) bool tensors: keep this token in the final text?
        for step in range(max_thinking_tokens + max_answer_tokens + 1):
            if print_steps:
                print(thinking)
            step_logits = logits / temperature
            probs = torch.softmax(step_logits, dim=-1)

            filtered_probs = self._filter_distribution(probs, top_k, top_p, min_p)
            sorted_probs, sorted_indices = self._compact_nonzero(filtered_probs)
            # print(sorted_probs)
            # print(sorted_indices)

            # next_id = torch.multinomial(filtered_probs, num_samples=1)  # (B, 1)
            next_id = probs.argmax(dim=-1, keepdim=True)

            is_think_end = torch.isin(next_id.squeeze(-1), self.think_end_id)
            is_eos = torch.isin(next_id.squeeze(-1), self.eos_ids)

            num_thinking_tokens = torch.where(thinking, num_thinking_tokens + 1, num_thinking_tokens)

            # entropy-based thinking -> answering transition, per row
            stop, _h = self._cold_stop_entropy(probs, cold_stop_threshold)
            bumped = torch.where(stop, low_entropy_streak + 1, torch.zeros_like(low_entropy_streak))
            low_entropy_streak = torch.where(thinking, bumped, low_entropy_streak)

            for b in range(B):
                if print_steps:
                    print(f"\n--- Step {step}, Batch {b} ---")
                    print(f"entropy: {_h}")

                for j in range(sorted_indices.shape[1]):
                    token_id = sorted_indices[b, j].item()
                    prob = sorted_probs[b, j].item()

                    token = self.tokenizer.decode(
                        [token_id],
                        skip_special_tokens=False
                    )
                    if print_steps:
                        print(f"{j:2d} | {prob:.6f} | {token_id:6d} | {repr(token)}")
                    trace.append({
                        "prompt": raw_prompts[b],
                        "batch": b,
                        "step": step,
                        "thinking": thinking[b].item(),
                        "entropy": _h[b].item(),
                        "next_id": next_id[b, 0].item(),
                        "topk_ids": sorted_indices[b].tolist(),
                        "topk_probs": sorted_probs[b].float().tolist(),
                    })


            max_thinking_reached = (thinking & (num_thinking_tokens >= max_thinking_tokens))
            should_switch = thinking & (is_think_end | (low_entropy_streak >= streak_threshold) | max_thinking_reached)
            low_entropy_streak = torch.where(should_switch, torch.zeros_like(low_entropy_streak), low_entropy_streak)

            if finished.all():
                break

            next_thinking = thinking & ~should_switch

            embed_indices = sorted_indices
            if mode == "shuffled" and sorted_indices.shape[1] > 1:
                # keep top-1, swap the other k-1 tokens for random ones (same weights)
                rand = torch.randint(0, self.embed_matrix.shape[0],
                                    sorted_indices[:, 1:].shape, device=self.device)
                embed_indices = torch.cat([sorted_indices[:, :1], rand], dim=1)

            soft_embed = self._soft_token_embedding_compact(sorted_probs, embed_indices)

            if mode == "norm":
                # rescale to the top-1 token's norm
                top1 = self.embed_matrix[sorted_indices[:, 0]].float()
                scale = top1.norm(dim=-1, keepdim=True) / soft_embed.float().norm(dim=-1, keepdim=True).clamp_min(1e-6)
                soft_embed = (soft_embed.float() * scale).to(self.embed_matrix.dtype)
            hard_embed = self.embed_matrix[next_id.squeeze(-1)].to(self.embed_matrix.dtype)
            think_end_embed = self.embed_matrix[self.think_end_id].expand(B, -1)

            next_embed = torch.where(next_thinking.unsqueeze(-1), soft_embed, hard_embed)
            next_embed = torch.where(should_switch.unsqueeze(-1), think_end_embed, next_embed)  # force it on the transition step

            answering_now = ((~next_thinking) & (~should_switch) & (~finished))
            num_answer_tokens = torch.where(answering_now, num_answer_tokens + 1, num_answer_tokens)
            finished = finished | (answering_now & is_eos)
            max_answer_reached = ((~next_thinking) & (~finished) & (num_answer_tokens >= max_answer_tokens))

            finished = finished | max_answer_reached

            thinking = next_thinking
            collected_ids.append(next_id)
            collected_mask.append(answering_now)

            attention_mask = torch.cat([attention_mask, attention_mask.new_ones((B, 1))], dim=-1)

            out = self.model(
                inputs_embeds=next_embed.unsqueeze(1),
                attention_mask=attention_mask,
                position_ids=next_position,
                past_key_values=past_key_values,
                use_cache=True,
            )
            past_key_values = out.past_key_values
            logits = out.logits[:, -1, :]
            next_position = next_position + 1

        # ---- assemble per-row text ----
        ids_tensor = torch.cat(collected_ids, dim=1)      # (B, T)
        mask_tensor = torch.stack(collected_mask, dim=1)  # (B, T)

        texts = []
        for i in range(B):
            row_ids = ids_tensor[i][mask_tensor[i]].tolist()
            texts.append(self.tokenizer.decode(row_ids, skip_special_tokens=True))
        return (texts, trace)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"
# model_name = "Qwen/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
)


In [ ]:
from datasets import load_dataset

gsm8k = load_dataset(
    "openai/gsm8k",
    "main",
    split="test"
)
print(gsm8k)
print(gsm8k[0]["question"])

In [5]:
import re

NUM = r"-?\d[\d,]*(?:\.\d+)?"

def _to_float(s):
    return float(s.replace(",", "").rstrip("."))

def extract_answer(text):
    """Returns (number or None, followed_format)."""
    text = text.replace("$", "")

    # 1. the requested format: \boxed{...}
    m = re.findall(rf"\\boxed\{{\s*({NUM})", text)
    if m:
        return _to_float(m[-1]), True

    # 2. alternatives: "#### 18" or "answer is 18"
    m = re.findall(rf"####\s*({NUM})", text) or \
        re.findall(rf"answer is[:\s]*\**\s*({NUM})", text, flags=re.I)
    if m:
        return _to_float(m[-1]), False

    # 3. last resort: last number in the text
    m = re.findall(NUM, text)
    return (_to_float(m[-1]), False) if m else (None, False)

def gold_answer(example):
    return _to_float(example["answer"].split("####")[-1].strip())

In [ ]:
import torch
from scipy.stats import binomtest
import json

SYSTEM_PROMPT = "Solve the following math problem. Put the final numeric answer in \\boxed{}."
BATCH_SIZE = 10
st = SoftThinking(model, tokenizer, device=None)

START, END = 0, 600
examples = gsm8k.shuffle(seed=0).select(range(START, END))
questions = [ex["question"] for ex in examples]
golds = [gold_answer(ex) for ex in examples]

configs = {
    "hard":     dict(top_k=1, mode="soft"),
    "soft":     dict(top_k=5, mode="soft"),
    "shuffled": dict(top_k=5, mode="shuffled"),
    "norm":     dict(top_k=5, mode="norm"),
}

name = "soft"
cfg = configs[name]

torch.manual_seed(0)

outs = []
with open(f"trace_{name}_{START}_{END}.jsonl", "w") as f:
    for i in range(0, len(questions), BATCH_SIZE):
        texts, trace = st.generate(questions[i:i + BATCH_SIZE],
                                   system_prompt=SYSTEM_PROMPT,
                                   max_thinking_tokens=1024,
                                   max_answer_tokens=512,
                                   **cfg)
        outs += texts
        for r in trace:
            r["question_idx"] = START + i + r.pop("batch")
            r["condition"] = name
            f.write(json.dumps(r) + "\n")
        f.flush()
        print(f"{name}: {len(outs)}/{len(questions)} done", flush=True)

preds = [extract_answer(o)[0] for o in outs]
correct = [p is not None and abs(p - g) < 1e-4 for p, g in zip(preds, golds)]

with open(f"results_{name}_{START}_{END}.json", "w") as f:
    json.dump({"questions": questions, "outputs": outs, "preds": preds,
               "golds": golds, "correct": correct}, f)
print(f"{name} accuracy: {sum(correct) / len(golds):.3f}", flush=True)